# Lesson 02 - மைக்ரோசாஃப்ட் ஏஜெண்ட் கட்டமைப்பை ஆராய்வு

**Microsoft Agent Framework (MAF)** என்பது AI ஏஜெண்டுகளை உருவாக்குவதற்கான ஒருங்கிணைந்த கட்டமைப்பாகும். இது நான்கு முக்கிய கூறுகளுடன் ஒரு சுத்தமான, இணைக்கக்கூடிய معماري வழங்குகிறது:

- **கிளையன்ட்** – ஒரு AI மாதிரி எண்ட்பாயிண்ட்டுடன் இணைந்து தொடர்பை கையாள்கிறது
- **ஏஜெண்ட்** – ஒரு கிளையண்டை அறிவுறுத்தல்கள் மற்றும் கருவி வரையறைகளுடன் சுற்றி கட்டுகிறது
- **கருவிகள்** – மாதிரி அழைக்கக்கூடிய தனிப்பயன் செயல்பாடுகளுடன் ஏஜெண்டின் திறன்களை விரிவுப்படுத்துகின்றன
- **அமர்வு** – பலதுறை உரையாடலுக்கான உரையாடல் வரலாற்றை பராமரிக்கிறது

இந்த பாடத்தில், நாம் இந்த கருத்துகளைக் கொண்டு **பயண முன்பதிவு ஏஜெண்ட்** ஒன்றை உருவாக்கப்போகின்றோம், இது இலக்கு கிடைக்கும் நிலையைச் சரிபார்க்கும்.


## அமைப்பு


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## முகவர் கட்டமைப்பு கட்டிடத்தின் புரிதல்

Microsoft முகவர் கட்டமைப்பு ஒரு படிகள் கொண்ட கட்டமைப்பை பின்பற்றுகிறது:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **கிளையண்ட்** – ஒரு `AzureAIProjectAgentProvider` Azure OpenAI பணியிடத்தை இணைக்கிறது. இது அங்கீகாரம், கோரிக்கை வடிவமைத்தல் மற்றும் பதில் பகுத்தறிதலை கையாள்கிறது.
2. **முகவர்** – `provider.create_agent()` மூலம் கிளையண்டிடமிருந்து உருவாக்கப்படுகிற முகவர், மாடல் அணுகலை (system prompt) மற்றும் கருவிகளுடன் இணைக்கிறது.
3. **கருவிகள்** – `@tool` என்ற அலங்கரம் உடன் கூடிய பைதான் செயல்பாடுகள், அவற்றை முகவர் செயல்படுத்த அல்லது தரவு பெற பயன்படுத்தலாம்.
4. **செஷன்** – ஒரு `AgentSession` பொருள் (agent.create_session() மூலம் உருவாக்கப்படும்) உரையாடல் வரலாற்றைச் சேமிக்கிறது, இது பல சுற்றுரையாடலில் முகவர் முன்னர் உள்ள உள்ளடக்கத்தை நினைவில் வைக்க உதவுகிறது.

ஒவ்வொரு படியையும் ஒன்றாக கட்டுவோம்.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool அலங்காரியுடன் கருவிகளை சேர்க்குதல்

கருவிகள் முகவரிகளுக்கு உரை உருவாக்கம் கடந்த செயல்களைச் செய்யவிடும். `@tool` அலங்காரியால் சாதாரண Python செயல்பாடு முகவரி அழைக்கக்கூடிய ஒன்றாக மாற்றப்படுகிறது.

முக்கியக் குறிப்புகள்:
- மாதிரிக்கு ஒவ்வொரு அளவுரும் புரியும் வகையில் `Annotated[type, "description"]` பயன்படுத்தவும்.
- டாக்ஸ்டிரிங் கருவி விவரமாக மாதிரி பார்க்கும்.
- `approval_mode="never_require"` என்றால் கருவி பயனர் ஒப்புதலின்றி தானாக இயங்கும்.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## கருவிகளுடன் ஒரு முகவர் உருவாக்குதல்

இப்போது நாங்கள் கிளையன்ட், வழிமுறைகள் மற்றும் கருவிகளை ஒரு முகவராக இணைக்கிறோம். `வழிமுறைகள்` என்பது கணினி அறிவுறுத்தல் ஆகும் — அவை முகவரின் தன்மையும் நடத்தையும் நிர்ணயிக்கின்றன.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## அம்சம்-முறை உரையாடல்கள் உடன் அமர்வுகள்

ஒரு `AgentSession` (`agent.create_session()` மூலமாக உருவாக்கப்பட்டது) உரையாடலின் அனைத்து செய்திகளையும் கண்காணிக்கிறது. ஒவ்வொரு `agent.run()` அழைப்புக்கும் அதே அமர்வை வழங்குவதன் மூலம், ஏஜெண்ட் முழு உரையாடல் வரலாறை அணுக முடியும் மற்றும் முன்னைய செய்திகளை மீண்டும் குறிப்பிட்டு பார்க்கலாம்.

ஒவ்வொரு முறையிலும் எங்கள் கிடைக்கும் நிலை சோதகையை அழைக்க ஏஜெண்டுக்கு அனுமதிக்க `tools=[check_destination_availability]` ஐ வழங்குகிறோம்.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework என்ற நான்கு தாண்டவக்கல்லுகளை ஆராய்ந்தீர்கள்:

| கருத்து | நீங்கள் கற்றுக்கொண்டது |
|---------|----------------------|
| **கிளையன்ட்** | `AzureAIProjectAgentProvider` அணுகல் அடிப்படையான அங்கீகாரத்துடன் Azure OpenAIயுடன் இணைக்கிறது |
| **ஏஜெண்ட்** | `provider.create_agent()` ஒரு மாதிரி இணைப்பை வழிமுறைகளும் பெயருமாக தொகுக்கிறது |
| **கருவிகள்** | `@tool` அலங்காரக்குறிப்பு Python செயல்பாடுகளை ஏஜெண்ட் அழைக்க வெளிப்படுத்துகிறது |
| **அமர்வு** | `agent.create_session()` பல முறைحوர்கள் முழுவதும் உரையாடல் வரலாறை பராமரிக்கிறது |

இவை சேர்ந்து இயங்குவதால் மூலமாக இயற்கையான உரையாடல்களை நடாத்தக்கூடிய, வெளிப்புற செயல்பாடுகளை அழைக்கக்கூடிய மற்றும் சூழலை பராமரிக்கக்கூடிய ஏஜெண்ட்கள் உருவாகின்றன — இது விரிவான ஏஜென்டிக் நுட்பங்களுக்கான அடித்தளம் ஆகும்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**குறிப்பு**:  
இந்த ஆவணம் AI மொழி மாற்ற சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழி பெயர்க்கப்பட்டது. நிதானமான தரம்பாணத்தை எங்கள் குழு உறுதி செய்கிறாலும், தானியங்கி மொழி மாற்றங்களில் பிழைகள் அல்லது தவறுகள் இருக்கக்கூடும் என்பதை தயவுசெய்து கவனத்தில் கொள்ளுங்கள். மூல ஆவணம் அதன் இயல்பான மொழியில் அதிகாரப்பூர்வமான ஆதாரமாகக் கொள்ளப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பின் பயன்பாட்டால் ஏற்படும் தவறான புரிதல்கள் அல்லது தவறான விளக்கங்களுக்காக நாங்கள் பொறுப்பேற்கவில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
